# Day 4, Notebook 2: the segment summary, and the count it rests on

Notebook 1 produced one number for the whole file. Nobody runs a business on one number.

The real question always sits one level down: which part of this is working, and how sure can anyone be. That means splitting the file into groups and computing the same honest statistic per group.

A second thing starts lying at that point, and it is not the statistic. It is the size of the group.

## Setup

Same file, same conversion, top of the notebook so this runs cold in a fresh Codespace.

`traceback` is here so a deliberate failure can print itself and let the notebook keep running. It is a teaching convenience and nothing more.

In [ ]:
import csv
import statistics
import traceback

CLEANED_CSV = "C2_W01_D04_data_cleaned_STUDENT.csv"

with open(CLEANED_CSV) as f:
    records = list(csv.DictReader(f))

for r in records:
    r["amount"] = int(r["amount"])

def show_failure(fn):
    # Run something expected to raise, print the trace, and keep the notebook alive.
    try:
        fn()
    except Exception:
        traceback.print_exc()

print("records loaded:", len(records))
print("segments present:", sorted({r["segment"] for r in records}))

## Where this is going, before we build any of it

The table you will have inside the hour:

```
segment      count   median amount   accepted rate
segment_d       12        Rs 6,750           58.3%
segment_c        6        Rs 6,800           50.0%
segment_a       20        Rs 7,950           45.0%
segment_b        9        Rs 8,000           44.4%
```

And then you will spend the rest of the session refusing to send it in that form.

## Section 1: counting into named piles

Monday you counted records that passed a test. One counter, one number out.

Today you need one counter per segment. The natural move is a dictionary with a counter behind each name.

```
   records                     counts
   -------                    --------
   segment_a  ---------->     segment_a : 1, 2, 3 ...
   segment_b  ---------->     segment_b : 1, 2 ...
   segment_a  ------|
   segment_d  ------|---->    segment_d : 1 ...
```

Here is the version a working trainer writes on the board from memory.

In [ ]:
def count_with_hardcoded_segments():
    counts = {"segment_a": 0, "segment_b": 0, "segment_c": 0}
    for r in records:
        counts[r["segment"]] = counts[r["segment"]] + 1
    return counts

show_failure(count_with_hardcoded_segments)

### The deliberate failure of this half

```
KeyError: 'segment_d'
```

Read it the way Tuesday taught you, from the bottom.

`KeyError` means a dictionary was handed a key it does not hold. Python then prints the key it did not hold, in quotes: `'segment_d'`.

The dictionary held three segments because a person typed three segments. The file holds four.

**Every hard-coded list of categories is a promise about data you have not read yet.** It survives exactly as long as nobody adds a new segment, and nobody tells you when they do.

### The fix: build the key the first time you see it

In [ ]:
counts = {}

for r in records:
    key = r["segment"]
    if key not in counts:
        counts[key] = 0
    counts[key] = counts[key] + 1

for key in sorted(counts):
    print("{}: {:>3}".format(key, counts[key]))
print("{:>9}: {:>3}".format("total", sum(counts.values())))

Four segments, discovered by reading rather than by remembering. The total matches your cleaned record count, and that last line is how you would catch a counting bug without being told there was one.

The same shape written shorter, which you have already met:

```python
counts[key] = counts.get(key, 0) + 1
```

That is Monday's `.get()` with a default, doing on Monday's terms exactly what the three-line version does. Both are correct. Use whichever your reader will understand faster.

## Section 2: three things per segment, in one pass

Per segment you want the count, a typical amount and the accepted share.

The count and the accepted share can be accumulated one record at a time. The median cannot: it needs every value in the group, sorted, before it can be taken. So the pass collects the amounts into a list and the median is computed after the loop finishes.

```
   one pass over records
          |
          +--> count      += 1
          +--> accepted   += 1 if outcome is accepted
          +--> amounts    .append(amount)
          |
     after the loop
          |
          +--> median = middle of sorted(amounts)
          +--> rate   = accepted / count
```

In [ ]:
def summarise_by(records, key_field):
    # Group records on key_field: count, median amount and accepted rate per group.
    buckets = {}
    for r in records:
        key = r[key_field]
        if key not in buckets:
            buckets[key] = {"count": 0, "amounts": [], "accepted": 0}
        buckets[key]["count"] += 1
        buckets[key]["amounts"].append(r["amount"])
        if r["outcome"] == "accepted":
            buckets[key]["accepted"] += 1

    summary = {}
    for key, b in buckets.items():
        summary[key] = {
            "count": b["count"],
            "median_amount": statistics.median(b["amounts"]),
            "accepted": b["accepted"],
            "rate": b["accepted"] / b["count"],
        }
    return summary

by_segment = summarise_by(records, "segment")

for key in sorted(by_segment):
    s = by_segment[key]
    print("{}  count {:>3}   median Rs {:>8,.0f}   accepted {:>2}   rate {:>5.1f}%".format(
        key, s["count"], s["median_amount"], s["accepted"], s["rate"] * 100))

Note the parameter. `summarise_by` takes the field to group on rather than hard-coding `"segment"`, which costs nothing today and is the only reason your take-home is a small job instead of a copy-paste job.

Note also what is absent from every row: the mean. Notebook 1 settled that, and the setting holds here.

## Section 3: the ranking, and what it is really measuring

Sort by rate, the way anybody would.

In [ ]:
ranked = sorted(by_segment.items(), key=lambda pair: pair[1]["rate"], reverse=True)

print("accepted rate, best first")
print("-" * 30)
for key, s in ranked:
    print("{}   {:>5.1f}%".format(key, s["rate"] * 100))

A clean ranking with a clear winner. `segment_d` leads by more than eight points.

Somebody in a meeting is now about to move budget. Print one more column before they do.

In [ ]:
print("accepted rate, best first, with the count it rests on")
print("-" * 52)
for key, s in ranked:
    print("{}   {:>5.1f}%   on {:>2} records".format(key, s["rate"] * 100, s["count"]))

### The trap, sprung

The two best-performing segments in the file are the two smallest segments in the file.

`segment_d` leads `segment_a` by 13.3 points on twelve records. Work out what that lead is worth.

In [ ]:
d = by_segment["segment_d"]
a = by_segment["segment_a"]

print("segment_d: {} accepted of {}  ->  {:.1f}%".format(d["accepted"], d["count"], d["rate"] * 100))
print("segment_a: {} accepted of {}  ->  {:.1f}%".format(a["accepted"], a["count"], a["rate"] * 100))
print()

for moved in range(0, 4):
    swung = (d["accepted"] - moved) / d["count"]
    print("if {} of segment_d's accepted records had gone the other way: {:.1f}%".format(moved, swung * 100))

Two records changing outcome erase the entire lead.

Two records. Out of a file of forty-seven, in a segment of twelve, on a difference somebody was about to fund.

This is not a flaw in these particular records. Small groups produce extreme rates in both directions, in any dataset, with nothing causing it. Rank a set of groups by a rate and the smallest groups drift towards both ends of the list, so the ranking reads group size as much as it reads performance.

The fix is not a cleverer statistic. It is the column you already printed.

## Section 4: the honest sentence

The deliverable of this session is not the table. A table gets screenshotted, cropped, and pasted into a slide by somebody who never saw your notebook.

The deliverable is a sentence per segment that survives that journey. Three parts, in this order:

```
[the number]   [the denominator]   [what it does not yet support]
```

The third part is the one that goes missing, so it goes in the sentence rather than in a footnote.

In [ ]:
TRUST_FLOOR = 30   # a threshold you should be prepared to defend, not a law of nature

for key in sorted(by_segment):
    s = by_segment[key]
    line = "{}: accepted on {} of {} records, {:.1f} percent.".format(
        key, s["accepted"], s["count"], s["rate"] * 100)
    if s["count"] < TRUST_FLOOR:
        line += " Fewer than {} records, so treat this as an indication rather than a measurement.".format(TRUST_FLOOR)
    print(line)
    print()

Every segment in this file trips the floor, which is itself the finding. A forty-seven record file does not support four confident segment claims, and saying so is the correct professional answer rather than a failure to produce one.

Written out for a person, with the money column handled the way notebook 1 settled:

```
segment_a: accepted on 9 of 20 records, 45.0 percent. The largest segment in the
           file and the steadiest number here.

segment_b: accepted on 4 of 9 records, 44.4 percent. Median order Rs 8,000. The
           mean of Rs 60,255.56 is one order and should not be quoted.

segment_c: accepted on 3 of 6 records, 50.0 percent. Six records support nothing.
           Treat as unmeasured.

segment_d: accepted on 7 of 12 records, 58.3 percent. Highest in the file, resting
           on twelve records; two different outcomes erase the lead.
```

Two of those four sentences decline to make a claim. That is the sentence doing its job.

### Milestone: what you can now answer

**Where this shows up in production.** Any dashboard that ranks stores, regions, cohorts or campaigns by a conversion rate has this problem built in, and the smallest units sit at the top and the bottom of the leaderboard month after month while the middle stays still. Teams that have been bitten once report the denominator beside every rate and grey out any row below a stated minimum. That minimum is a judgement call somebody wrote down and defended, and being the person who writes it down is the job.

**Interview question this section just made answerable.**

> Segment A converts at 42 percent on 12 records and segment B at 31 percent on 1,200. Which do you trust?

Trust segment B's number. Say why in terms of movement: one record flips segment A's rate by more than eight points, while one record moves segment B by less than a tenth of a point. Then add the part most candidates leave out, which is that segment A is not reported as bad or good, it is reported as unmeasured, and you would say what volume it needs before the number means anything.

## The question this session does not answer

Is `segment_d` genuinely better than `segment_a`, or is 58.3 against 45.0 what four groups of these sizes do on their own?

That question has a real answer and a method behind it. It is Monday's session.

Today you name it, write it down, and stop. Write this line into your own notebook now, because you will open it on Monday:

> Open question: segment_d 58.3 percent on 12 records against segment_a 45.0 percent on 20 records. Is that gap real? Method needed to settle it.

Saying "we do not know yet, and here is exactly what would tell us" is a complete answer in a room full of people who wanted a different one.

## What this notebook settled

| Question | Answer |
|---|---|
| How do you group without knowing the categories | Build the key on first sight; a hard-coded list gives you `KeyError: 'segment_d'` |
| Which statistic per segment | Median amount, because the money column has a tail |
| What goes beside every rate | The count it rests on, on the same line |
| Which segments lead on rate | The two smallest ones, `segment_d` on 12 and `segment_c` on 6 |
| Is the gap real | Unanswered today by design, and named as Monday's work |

**Crux.** Every rate carries its denominator, or it lies for you while you are not in the room.

Week 2 re-expresses this exact pass as one line of pandas and one SQL `GROUP BY`. You built it by hand once so that when the one-liner arrives you already know what the answer should be, and you will notice if it disagrees.